# Step 6 — Pareto Front Mapping (Calculations P1–P4)

**Purpose:** Map the empirical Pareto front in the 3D objective space (Φ_S, Φ_A, Φ_R) for all memory sequences.

- **P1** — Global Pareto front: compute (Φ_S, Φ_A, Φ_R) triplet per sequence; non-dominated sorting on a stratified sample; 2D projection figures.
- **P2** — Per-germline fronts: exact Pareto rank per IGHV gene; front statistics and hypervolume proxy.
- **P3** — Per-isotype fronts: front shift IgM → IgG → IgA, consistent with maturation hypothesis.
- **P4** — Per-donor fronts: inter-individual variation in front shape and spread.

## Theoretical basis

The maturation optimum lies on the Pareto front of the three objectives:
```
min (Φ_A, Φ_S, Φ_R)    subject to Φ_S ≤ c_S, Φ_R ≤ c_R
```
A memory sequence x is **Pareto-optimal** if no other sequence y satisfies:
```
Φ_A(y) ≤ Φ_A(x)  AND  Φ_S(y) ≤ Φ_S(x)  AND  Φ_R(y) ≤ Φ_R(x)  (with at least one strict)
```
Lower values of all three objectives are better:
- Φ_A ↓ : higher affinity selection signal (more CDR replacements, fewer FWR)
- Φ_S ↓ : fewer structurally costly FWR mutations
- Φ_R ↓ : lower autoreactivity risk (safer CDRH3 chemistry)

## Computational strategy

For N = 1.46M sequences in 3D, exact non-dominated sorting is computationally expensive (O(N² k)).
We use a **stratified approach**:
- **P1 global**: stratified random sample of 200K sequences (50K per isotype, all IgE due to small count); exact Pareto rank 1 computed on sample.
- **P2 per-germline**: exact Pareto rank 1 within each germline (max ~200K sequences per germline; use sort-sweep algorithm O(N × H) where H = front size ≪ N).
- **P3 per-isotype**: exact Pareto rank 1 within each isotype (subsample to 100K for IgM which is ~839K).
- **P4 per-donor**: exact Pareto rank 1 within each donor (max ~300K sequences).

**Inputs:** `results/tables/omega_per_position.parquet`, `results/tables/affinity_proxy.parquet`, `results/tables/phi_r_scores.parquet`  
**Outputs:** `results/tables/pareto_*.csv`, `results/figures/fig_p*.png`

In [ ]:
import polars as pl
import numpy as np
import bisect
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

from scipy.stats import spearmanr

In [ ]:
DATA_DIR = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
RESULTS  = DATA_DIR / "results"
FIGURES  = RESULTS / "figures"
TABLES   = RESULTS / "tables"

print("Paths OK")

In [ ]:
# ── Pareto rank-1 helper (3D minimisation, sort-sweep algorithm) ───────────────
# Algorithm: sort by obj0; sweep in that order; maintain a running 2D Pareto
# staircase (sorted by obj1 asc, obj2 strictly decreasing) for all previously
# seen points. A new point is dominated iff there is a staircase entry with
# obj1 ≤ c1 AND obj2 ≤ c2. O(N × H) where H = front size (typically << N).

def find_pareto_front_3d(costs: np.ndarray) -> np.ndarray:
    """
    Find rank-1 Pareto-optimal points for 3D minimisation.
    costs : (N, 3) float array.
    Returns : boolean mask of length N.
    """
    n = len(costs)
    order = np.argsort(costs[:, 0], kind='stable')
    sc = costs[order]
    is_front = np.zeros(n, dtype=bool)

    # 2D running staircase in (obj1, obj2): obj1 sorted ascending, obj2 monotone decreasing.
    f1 = []   # list of float
    f2 = []   # list of float

    for k in range(n):
        c1, c2 = sc[k, 1], sc[k, 2]

        # Find position in staircase where obj1 > c1
        pos = bisect.bisect_right(f1, c1)
        # If there is a staircase entry in [0..pos-1], its min obj2 is f2[pos-1]
        # (staircase is monotone decreasing → last element in range has smallest obj2)
        if pos > 0 and f2[pos - 1] <= c2:
            continue   # dominated

        # Not dominated
        is_front[order[k]] = True

        # Update staircase: remove points dominated by (c1, c2)
        # A staircase point (f1_j, f2_j) is dominated by (c1, c2) iff c1 ≤ f1_j AND c2 ≤ f2_j
        # Since f1 is ascending, dominated candidates start at pos (f1_j ≥ c1 → f1_j > c1 since
        # equal would be at pos too). Remove consecutive entries while f2[pos] >= c2.
        while pos < len(f1) and f2[pos] >= c2:
            f1.pop(pos)
            f2.pop(pos)
        f1.insert(pos, c1)
        f2.insert(pos, c2)

    return is_front


def front_stats(costs_front: np.ndarray) -> dict:
    """Summary statistics for a Pareto front (N_front x 3 array)."""
    return {
        'n_front':    len(costs_front),
        'mean_phi_S': float(np.mean(costs_front[:, 0])),
        'mean_phi_A': float(np.mean(costs_front[:, 1])),
        'mean_phi_R': float(np.mean(costs_front[:, 2])),
        'min_phi_A':  float(np.min(costs_front[:, 1])),
        'min_phi_R':  float(np.min(costs_front[:, 2])),
        'min_phi_S':  float(np.min(costs_front[:, 0])),
    }


print("Pareto helper functions defined.")

In [ ]:
# ── Load and compute objective triplets (Φ_S, Φ_A, Φ_R) ─────────────────────
# Φ_A: from affinity_proxy.parquet (Step 3)
# Φ_R: from phi_r_scores.parquet (Step 4)
# Φ_S: recomputed here from omega_per_position.parquet (Step 2) × mutation counts
#   Φ_S(x) = n_mut_CDR_H × ⟨φ_S⟩_CDR + n_mut_FWR_H × ⟨φ_S⟩_FWR
#   where ⟨φ_S⟩_region = mean_i[max(0, -ln ω_i)] for positions i in that region.

# --- Load omega → compute regional phi_S costs ---
omega_df = pl.read_parquet(TABLES / "omega_per_position.parquet")

omega_df = omega_df.with_columns(
    pl.when(pl.col('omega').is_not_null() & (pl.col('omega') > 0))
    .then((-pl.col('omega').log(math.e)).clip(lower_bound=0.0))
    .otherwise(0.0)
    .alias('phi_s_local')
)

region_phi_s = (
    omega_df.group_by('region')
    .agg(pl.col('phi_s_local').mean().alias('mean_phi_s'))
    .to_dicts()
)
rd = {r['region']: r['mean_phi_s'] for r in region_phi_s}

PHI_S_CDR = float(np.mean([rd.get('CDR1', 0.0), rd.get('CDR2', 0.0)]))
PHI_S_FWR = float(np.mean([rd.get('FR1', 0.0), rd.get('FR2', 0.0), rd.get('FR3', 0.0)]))

print(f"⟨φ_S⟩_CDR = {PHI_S_CDR:.4f}  |  ⟨φ_S⟩_FWR = {PHI_S_FWR:.4f}")

# --- Load phi_A and phi_R, join, compute phi_S ---
phi_a_df = pl.read_parquet(TABLES / "affinity_proxy.parquet")
phi_r_df = pl.read_parquet(TABLES / "phi_r_scores.parquet")

data = (
    phi_a_df
    .filter(pl.col('phi_A').is_not_null())
    .select(['seq_name', 'v_gene:0', 'isotype_class', 'donor', 'lineage',
             'n_R_CDR_H', 'n_S_CDR_H', 'n_R_FWR_H', 'n_S_FWR_H', 'phi_A'])
    .join(phi_r_df.select(['seq_name', 'phi_R']), on='seq_name', how='inner')
    .with_columns([
        (pl.col('n_R_CDR_H') + pl.col('n_S_CDR_H')).alias('n_mut_CDR_H'),
        (pl.col('n_R_FWR_H') + pl.col('n_S_FWR_H')).alias('n_mut_FWR_H'),
    ])
    .with_columns(
        (pl.col('n_mut_CDR_H') * PHI_S_CDR + pl.col('n_mut_FWR_H') * PHI_S_FWR).alias('phi_S')
    )
)

print(f"Dataset: {data.height:,} sequences, {data.width} columns")
print(f"  phi_A: mean={data['phi_A'].mean():.3f}  std={data['phi_A'].std():.3f}")
print(f"  phi_S: mean={data['phi_S'].mean():.3f}  std={data['phi_S'].std():.3f}")
print(f"  phi_R: mean={data['phi_R'].mean():.3f}  std={data['phi_R'].std():.3f}")
print(f"  Isotypes: {sorted(data['isotype_class'].unique().to_list())}")
print(f"  V-genes:  {data['v_gene:0'].n_unique()}")
print(f"  Donors:   {data['donor'].n_unique()}")

In [ ]:
# ── Normalize objectives to [0, 1] (min-max across all memory sequences) ─────
# This makes objectives comparable and is required for hypervolume computation.

phi_A_min, phi_A_max = float(data['phi_A'].min()), float(data['phi_A'].max())
phi_S_min, phi_S_max = float(data['phi_S'].min()), float(data['phi_S'].max())
phi_R_min, phi_R_max = float(data['phi_R'].min()), float(data['phi_R'].max())

print(f"Normalisation ranges:")
print(f"  phi_A: [{phi_A_min:.3f}, {phi_A_max:.3f}]")
print(f"  phi_S: [{phi_S_min:.3f}, {phi_S_max:.3f}]")
print(f"  phi_R: [{phi_R_min:.3f}, {phi_R_max:.3f}]")

data = data.with_columns([
    ((pl.col('phi_A') - phi_A_min) / (phi_A_max - phi_A_min)).alias('phi_A_n'),
    ((pl.col('phi_S') - phi_S_min) / (phi_S_max - phi_S_min)).alias('phi_S_n'),
    ((pl.col('phi_R') - phi_R_min) / (phi_R_max - phi_R_min)).alias('phi_R_n'),
])

# Save full triplets table (parquet; large)
data.write_parquet(TABLES / "objective_triplets.parquet")
print(f"Saved → objective_triplets.parquet ({data.height:,} rows)")

## P1 — Global Pareto Front (stratified sample)

For N = 1.46M sequences exact non-dominated sorting is O(N²k) in the worst case.  
We use a **stratified sample** of 200K sequences (50K per isotype; all IgE if < 50K)  
to characterise the global front shape, followed by exact per-germline / per-isotype / per-donor fronts in P2–P4.

The 3D Pareto rank-1 is computed using the `find_pareto_front_3d` sort-sweep helper (O(N × H)) defined above.

In [ ]:
# ── P1: stratified sample + global Pareto rank 1 ─────────────────────────────
SAMPLE_PER_ISO = 50_000
rng = np.random.default_rng(42)

sample_parts = []
for iso, sub in data.partition_by('isotype_class', as_dict=True).items():
    iso_str = iso[0] if isinstance(iso, (list, tuple)) else str(iso)
    n_iso = sub.height
    n_take = min(n_iso, SAMPLE_PER_ISO)
    idx = rng.choice(n_iso, size=n_take, replace=False)
    sample_parts.append(sub[idx])
    print(f"  {iso_str}: {n_iso:>9,} sequences → sampled {n_take:,}")

sample_df = pl.concat(sample_parts)
print(f"\nStratified sample: {sample_df.height:,} sequences")

# Compute Pareto rank 1 on sample (normalised objectives)
costs_s = sample_df.select(['phi_S_n', 'phi_A_n', 'phi_R_n']).to_numpy()
is_front_s = find_pareto_front_3d(costs_s)

sample_df = sample_df.with_columns(
    pl.Series('pareto_rank1', is_front_s.astype(int))
)

n_front = int(is_front_s.sum())
print(f"\nPareto rank-1 sequences: {n_front:,} / {sample_df.height:,} "
      f"({100*n_front/sample_df.height:.1f}%)")

# Save sample Pareto table
(
    sample_df
    .select(['seq_name', 'isotype_class', 'v_gene:0', 'donor',
             'phi_S', 'phi_A', 'phi_R', 'phi_S_n', 'phi_A_n', 'phi_R_n', 'pareto_rank1'])
    .write_csv(TABLES / "pareto_sample.csv")
)
print(f"Saved → pareto_sample.csv")

In [ ]:
# ── P1 plot: 2D projections of global Pareto front ───────────────────────────
# Three 2D projections: phi_S vs phi_A | phi_S vs phi_R | phi_A vs phi_R
# Pareto front members highlighted in orange; background in blue.

front_mask = sample_df['pareto_rank1'].to_numpy().astype(bool)
bg   = sample_df.filter(pl.col('pareto_rank1') == 0)
frt  = sample_df.filter(pl.col('pareto_rank1') == 1)

PAIRS = [
    ('phi_S_n', 'phi_A_n', 'Φ_S (normalised)', 'Φ_A (normalised)'),
    ('phi_S_n', 'phi_R_n', 'Φ_S (normalised)', 'Φ_R (normalised)'),
    ('phi_A_n', 'phi_R_n', 'Φ_A (normalised)', 'Φ_R (normalised)'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (xc, yc, xl, yl) in zip(axes, PAIRS):
    ax.scatter(bg[xc].to_numpy(),  bg[yc].to_numpy(),
               s=0.5, alpha=0.03, color='#1E88E5', rasterized=True)
    ax.scatter(frt[xc].to_numpy(), frt[yc].to_numpy(),
               s=2, alpha=0.4, color='#FF6F00', rasterized=True)
    ax.set_xlabel(xl, fontsize=9)
    ax.set_ylabel(yl, fontsize=9)
    ax.set_title(f'{xl.split()[0]} vs {yl.split()[0]}')

legend_handles = [
    mpatches.Patch(color='#1E88E5', label=f'Non-Pareto (n={len(bg):,})', alpha=0.6),
    mpatches.Patch(color='#FF6F00', label=f'Pareto rank-1 (n={len(frt):,})', alpha=0.8),
]
axes[0].legend(handles=legend_handles, fontsize=8)
fig.suptitle(f'Global Pareto front — 3D objective space (stratified sample n={sample_df.height:,})',
             fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES / "fig_p1_pareto_projections.png", dpi=150, bbox_inches='tight')

# CSV: save Pareto front members' raw objective values
frt.select(['seq_name', 'isotype_class', 'v_gene:0', 'donor',
            'phi_S_n', 'phi_A_n', 'phi_R_n']).write_csv(
    FIGURES / "fig_p1_pareto_projections.csv"
)

plt.show()
print("Saved.")

## P2 — Per-Germline Pareto Fronts

For each IGHV gene (≥ 1 000 sequences), compute the exact Pareto rank-1 front and record front statistics:
- Front size (number of non-dominated sequences, raw and as % of stratum)
- Front centroid (mean Φ_A, Φ_S, Φ_R on the front)
- Hypervolume proxy: area under the 2D Pareto curve in the (Φ_S_n, Φ_A_n) projection, approximated by the **dominated hypervolume** relative to a reference point (1.1, 1.1).

**Key questions:**
- Do IGHV1-2 and IGHV1-69 (bnAb germlines) have smaller, higher-quality fronts (lower mean Φ_A on front)?
- Does IGHV4-34 (autoreactive) show a front shifted toward higher Φ_R?

In [ ]:
# ── P2 per-germline Pareto fronts ─────────────────────────────────────────────
MIN_N_GERM = 1000
MAX_N_GERM = 200_000   # cap for compute time; subsample larger germlines

BNAB_GENES   = {'IGHV1-2', 'IGHV1-69', 'IGHV1-69-2'}
AUTOREACTIVE = {'IGHV4-34'}

germline_pareto = []

for vk, sub in data.partition_by('v_gene:0', as_dict=True).items():
    vgene = vk[0] if isinstance(vk, (list, tuple)) else str(vk)
    n = sub.height
    if n < MIN_N_GERM:
        continue

    # Subsample if very large
    if n > MAX_N_GERM:
        idx = rng.choice(n, size=MAX_N_GERM, replace=False)
        sub = sub[idx]

    costs = sub.select(['phi_S_n', 'phi_A_n', 'phi_R_n']).to_numpy()
    is_f  = find_pareto_front_3d(costs)
    front_costs = costs[is_f]

    # 2D hypervolume approximation in (phi_S_n, phi_A_n) using dominated area
    # Sort front by phi_S_n ascending; compute step-function dominated area
    # relative to reference point (1.1, 1.1)
    ref = 1.1
    if len(front_costs) > 1:
        f2d = front_costs[:, :2]   # (phi_S_n, phi_A_n)
        ord2 = np.argsort(f2d[:, 0])
        f2d  = f2d[ord2]
        # Ensure monotone (remove dominated in 2D projection)
        min_a = np.minimum.accumulate(f2d[:, 1][::-1])[::-1]
        keep  = f2d[:, 1] == min_a
        f2d   = f2d[keep]
        # Dominated area (maximising): sweeping from left
        hv2d  = 0.0
        prev_s = 0.0
        prev_a = ref
        for s_val, a_val in f2d:
            hv2d += (s_val - prev_s) * (prev_a - a_val)
            prev_s = s_val
            prev_a = a_val
        hv2d += (ref - prev_s) * (prev_a - 0.0)   # right remainder
    else:
        hv2d = 0.0

    stats = front_stats(front_costs)
    germline_pareto.append({
        'v_gene':        vgene,
        'n_total':       n,
        'n_front':       int(is_f.sum()),
        'pct_front':     float(100 * is_f.sum() / len(is_f)),
        'hv2d_proxy':    float(hv2d),
        'mean_phi_S':    stats['mean_phi_S'],
        'mean_phi_A':    stats['mean_phi_A'],
        'mean_phi_R':    stats['mean_phi_R'],
        'min_phi_A':     stats['min_phi_A'],
        'min_phi_R':     stats['min_phi_R'],
        'is_bnab':       vgene in BNAB_GENES,
        'is_autoreact':  vgene in AUTOREACTIVE,
    })

gp_df = pl.DataFrame(germline_pareto).sort('hv2d_proxy', descending=True)
gp_df.write_csv(TABLES / "pareto_by_germline.csv")

print(f"Germlines analysed: {len(germline_pareto)}")
print(f"\nTop 10 by 2D hypervolume proxy:")
print(gp_df.head(10))
print(f"\nBnAb germlines:")
print(gp_df.filter(pl.col('is_bnab')))
print(f"\nIGHV4-34:")
print(gp_df.filter(pl.col('v_gene') == 'IGHV4-34'))
print(f"\nSaved → pareto_by_germline.csv")

In [ ]:
# ── P2 plot: per-germline hypervolume proxy (ranked bar chart) ────────────────
top_n = min(40, gp_df.height)
gp_plot = gp_df.head(top_n)

hv_vals  = gp_plot['hv2d_proxy'].to_numpy()
g_names  = gp_plot['v_gene'].to_list()
g_colors = ['#FF6F00' if r else '#4CAF50' if b else '#1E88E5'
            for r, b in zip(gp_plot['is_autoreact'].to_list(),
                            gp_plot['is_bnab'].to_list())]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(top_n), hv_vals[::-1], color=g_colors[::-1], alpha=0.8)
ax.set_yticks(range(top_n))
ax.set_yticklabels(g_names[::-1], fontsize=7)
ax.set_xlabel('2D Hypervolume proxy (Φ_S–Φ_A projection)', fontsize=9)
ax.set_title(f'Per-germline Pareto front quality (top {top_n} by hypervolume proxy)\n'
             f'Higher = larger dominated area = better affinity–structure trade-off', fontsize=10)

legend_handles = [
    mpatches.Patch(color='#FF6F00', label='IGHV4-34 (autoreactive)'),
    mpatches.Patch(color='#4CAF50', label='bnAb germlines (IGHV1-2, IGHV1-69)'),
    mpatches.Patch(color='#1E88E5', label='Other germlines'),
]
ax.legend(handles=legend_handles, fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "fig_p2_germline_hypervolume.png", dpi=150, bbox_inches='tight')

gp_plot.select(['v_gene', 'hv2d_proxy', 'n_front', 'pct_front',
                'mean_phi_A', 'mean_phi_S', 'mean_phi_R']).write_csv(
    FIGURES / "fig_p2_germline_hypervolume.csv"
)

plt.show()
print("Saved.")

In [ ]:
# ── P2 plot: overlaid Pareto fronts for top 6 germlines (Φ_S vs Φ_A) ─────────
# Select 4 top-HV ordinary germlines + IGHV1-2 + IGHV4-34 for comparison.
TOP_GENES = (
    gp_df.filter(~pl.col('is_bnab') & ~pl.col('is_autoreact'))
         .head(4)['v_gene'].to_list()
)
SHOW_GENES = TOP_GENES + ['IGHV1-2', 'IGHV4-34']
# Remove any that don't exist in data
SHOW_GENES = [g for g in SHOW_GENES if data.filter(pl.col('v_gene:0') == g).height > 0]

cmap  = plt.cm.get_cmap('tab10', len(SHOW_GENES))
COLORS_GEN = {g: cmap(i) for i, g in enumerate(SHOW_GENES)}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for gname in SHOW_GENES:
    sub = data.filter(pl.col('v_gene:0') == gname)
    if sub.height == 0:
        continue
    n_sub = sub.height
    if n_sub > MAX_N_GERM:
        idx = rng.choice(n_sub, size=MAX_N_GERM, replace=False)
        sub = sub[idx]

    costs = sub.select(['phi_S_n', 'phi_A_n', 'phi_R_n']).to_numpy()
    is_f  = find_pareto_front_3d(costs)
    fc    = costs[is_f]

    # Sort front by phi_S_n for line plot
    sort_idx = np.argsort(fc[:, 0])
    fc_sorted = fc[sort_idx]

    col = COLORS_GEN[gname]
    ls  = '--' if gname in AUTOREACTIVE else ('-' if gname in BNAB_GENES else ':')
    lw  = 2.0 if gname in (BNAB_GENES | AUTOREACTIVE) else 1.0

    axes[0].scatter(fc[:, 0], fc[:, 1], s=4, alpha=0.5, color=col)
    axes[0].plot(fc_sorted[:, 0], fc_sorted[:, 1], color=col, lw=lw, ls=ls,
                 label=gname)

    axes[1].scatter(fc[:, 1], fc[:, 2], s=4, alpha=0.5, color=col)

axes[0].set_xlabel('Φ_S (normalised)', fontsize=9)
axes[0].set_ylabel('Φ_A (normalised)', fontsize=9)
axes[0].set_title('Per-germline Pareto fronts\n(Φ_S vs Φ_A, lower-left = better)')
axes[0].legend(fontsize=7, ncol=2)

axes[1].set_xlabel('Φ_A (normalised)', fontsize=9)
axes[1].set_ylabel('Φ_R (normalised)', fontsize=9)
axes[1].set_title('Per-germline Pareto fronts\n(Φ_A vs Φ_R)')

plt.tight_layout()
plt.savefig(FIGURES / "fig_p2_germline_fronts.png", dpi=150, bbox_inches='tight')

# CSV: concatenate front coordinates for each gene
front_records = []
for gname in SHOW_GENES:
    sub = data.filter(pl.col('v_gene:0') == gname)
    if sub.height == 0:
        continue
    n_sub = sub.height
    if n_sub > MAX_N_GERM:
        idx = rng.choice(n_sub, size=MAX_N_GERM, replace=False)
        sub = sub[idx]
    costs = sub.select(['phi_S_n', 'phi_A_n', 'phi_R_n']).to_numpy()
    is_f  = find_pareto_front_3d(costs)
    for row in costs[is_f]:
        front_records.append({'v_gene': gname, 'phi_S_n': row[0],
                              'phi_A_n': row[1], 'phi_R_n': row[2]})

pl.DataFrame(front_records).write_csv(FIGURES / "fig_p2_germline_fronts.csv")
plt.show()
print("Saved.")

## P3 — Per-Isotype Pareto Fronts

If maturation is progressive (IgM → IgG → IgA in terms of SHM depth, per Step 1 findings),  
the Pareto fronts should shift: IgG / IgA sequences should extend further into the low-Φ_A  
(high-affinity) region at the cost of higher Φ_S (more structural penalty).

**Expected pattern:**
- IgM front: near the germline corner (low Φ_S, moderate–high Φ_A) — not yet selected
- IgG front: extended into low Φ_A region — affinity-driven selection
- IgA front: similar to IgG but potentially more Φ_R-constrained (gut mucosa: T-independent)
- IgE front: furthest in low Φ_A (deepest maturation, fewest sequences)

In [ ]:
# ── P3 per-isotype Pareto fronts ──────────────────────────────────────────────
SAMPLE_ISO = 100_000   # cap for large isotype strata

isotype_pareto = []
isotype_front_data = {}   # for plotting

for ik, sub in data.partition_by('isotype_class', as_dict=True).items():
    iso = ik[0] if isinstance(ik, (list, tuple)) else str(ik)
    n = sub.height
    sampled = min(n, SAMPLE_ISO)
    if n > SAMPLE_ISO:
        idx = rng.choice(n, size=SAMPLE_ISO, replace=False)
        sub_s = sub[idx]
    else:
        sub_s = sub

    costs = sub_s.select(['phi_S_n', 'phi_A_n', 'phi_R_n']).to_numpy()
    is_f  = find_pareto_front_3d(costs)
    front_costs = costs[is_f]

    # 2D HV proxy
    ref = 1.1
    hv2d = 0.0
    if len(front_costs) > 1:
        f2d = front_costs[:, :2]
        ord2 = np.argsort(f2d[:, 0])
        f2d  = f2d[ord2]
        min_a = np.minimum.accumulate(f2d[:, 1][::-1])[::-1]
        keep = f2d[:, 1] == min_a
        f2d = f2d[keep]
        prev_s, prev_a = 0.0, ref
        for s_val, a_val in f2d:
            hv2d += (s_val - prev_s) * (prev_a - a_val)
            prev_s, prev_a = s_val, a_val
        hv2d += (ref - prev_s) * (prev_a - 0.0)

    stats = front_stats(front_costs)
    isotype_pareto.append({
        'isotype':     iso,
        'n_total':     n,
        'n_sampled':   sampled,
        'n_front':     int(is_f.sum()),
        'pct_front':   float(100 * is_f.sum() / len(is_f)),
        'hv2d_proxy':  float(hv2d),
        **{k: stats[k] for k in stats},
    })
    isotype_front_data[iso] = front_costs
    print(f"  {iso}: n={n:>9,} → sampled {sampled:,} → front {int(is_f.sum()):,} "
          f"({100*is_f.sum()/len(is_f):.1f}%)  HV={hv2d:.4f}")

ip_df = pl.DataFrame(isotype_pareto)
ip_df.write_csv(TABLES / "pareto_by_isotype.csv")
print(f"\n{ip_df}")
print(f"\nSaved → pareto_by_isotype.csv")

In [ ]:
# ── P3 plot: isotype Pareto fronts (Φ_S vs Φ_A, Φ_A vs Φ_R) ─────────────────
ISO_ORDER  = ['IgM', 'IgG', 'IgA', 'IgE']
ISO_COLORS = {'IgM': '#607D8B', 'IgG': '#1E88E5', 'IgA': '#43A047', 'IgE': '#E53935'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for iso in ISO_ORDER:
    if iso not in isotype_front_data:
        continue
    fc  = isotype_front_data[iso]
    col = ISO_COLORS.get(iso, 'gray')
    # Sort by phi_S for step-line
    ord_s = np.argsort(fc[:, 0])
    fc_s  = fc[ord_s]

    axes[0].scatter(fc[:, 0], fc[:, 1], s=4, alpha=0.4, color=col)
    axes[0].plot(fc_s[:, 0], fc_s[:, 1], color=col, lw=2, label=iso)

    axes[1].scatter(fc[:, 1], fc[:, 2], s=4, alpha=0.4, color=col)
    ord_a = np.argsort(fc[:, 1])
    axes[1].plot(fc[ord_a, 1], fc[ord_a, 2], color=col, lw=2, label=iso)

for ax, xl, yl, title in [
    (axes[0], 'Φ_S (normalised)', 'Φ_A (normalised)', 'Structural cost vs Affinity deficit'),
    (axes[1], 'Φ_A (normalised)', 'Φ_R (normalised)', 'Affinity deficit vs Reactivity risk'),
]:
    ax.set_xlabel(xl, fontsize=9)
    ax.set_ylabel(yl, fontsize=9)
    ax.set_title(title + '\n(lower-left = better)', fontsize=9)
    ax.legend(fontsize=9)

fig.suptitle('Per-isotype Pareto fronts — maturation progression', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / "fig_p3_isotype_fronts.png", dpi=150, bbox_inches='tight')

# CSV
iso_recs = []
for iso, fc in isotype_front_data.items():
    for row in fc:
        iso_recs.append({'isotype': iso, 'phi_S_n': row[0], 'phi_A_n': row[1], 'phi_R_n': row[2]})
pl.DataFrame(iso_recs).write_csv(FIGURES / "fig_p3_isotype_fronts.csv")

plt.show()
print("Saved.")

## P4 — Per-Donor Pareto Fronts

Compute Pareto rank-1 front for each donor (≥ 5 000 memory sequences).  
If individuals differ in their maturation boundary (tolerance, infection history, genetics),  
we expect inter-individual variation in:
- **Front hypervolume**: larger HV = more diverse strategies explored
- **Front centroid**: different mean Φ_A (average affinity gain) on the front
- **λ_S / λ_R correlation**: donors with high λ_R (from Step 5) should have fronts shifted toward low Φ_R

In [ ]:
# ── P4 per-donor Pareto fronts ────────────────────────────────────────────────
MIN_DONOR_N = 5000

donor_pareto = []
donor_front_data = {}

for dk, sub in data.partition_by('donor', as_dict=True).items():
    donor = dk[0] if isinstance(dk, (list, tuple)) else str(dk)
    n = sub.height
    if n < MIN_DONOR_N:
        continue

    costs = sub.select(['phi_S_n', 'phi_A_n', 'phi_R_n']).to_numpy()
    is_f  = find_pareto_front_3d(costs)
    front_costs = costs[is_f]

    # 2D HV proxy
    ref = 1.1
    hv2d = 0.0
    if len(front_costs) > 1:
        f2d  = front_costs[:, :2]
        ord2 = np.argsort(f2d[:, 0])
        f2d  = f2d[ord2]
        min_a = np.minimum.accumulate(f2d[:, 1][::-1])[::-1]
        keep = f2d[:, 1] == min_a
        f2d = f2d[keep]
        prev_s, prev_a = 0.0, ref
        for s_val, a_val in f2d:
            hv2d += (s_val - prev_s) * (prev_a - a_val)
            prev_s, prev_a = s_val, a_val
        hv2d += (ref - prev_s) * (prev_a - 0.0)

    stats = front_stats(front_costs)
    donor_pareto.append({
        'donor':      donor,
        'n_total':    n,
        'n_front':    int(is_f.sum()),
        'pct_front':  float(100 * is_f.sum() / len(is_f)),
        'hv2d_proxy': float(hv2d),
        **{k: stats[k] for k in stats},
    })
    donor_front_data[donor] = front_costs
    print(f"  {donor}: n={n:>7,} → front {int(is_f.sum()):>4,} ({100*is_f.sum()/len(is_f):.1f}%)  "
          f"HV={hv2d:.4f}")

dp_df = pl.DataFrame(donor_pareto).sort('donor')
dp_df.write_csv(TABLES / "pareto_by_donor.csv")

print(f"\nHV proxy across donors:")
print(f"  mean={dp_df['hv2d_proxy'].mean():.4f}  std={dp_df['hv2d_proxy'].std():.4f}  "
      f"range=[{dp_df['hv2d_proxy'].min():.4f}, {dp_df['hv2d_proxy'].max():.4f}]")
print(f"\nSaved → pareto_by_donor.csv")

In [ ]:
# ── P4 plot: per-donor front hypervolume + Φ_A centroid ──────────────────────
donors_sorted = dp_df.sort('hv2d_proxy', descending=True)['donor'].to_list()
hv_d     = dp_df.sort('hv2d_proxy', descending=True)['hv2d_proxy'].to_numpy()
mean_A_d = dp_df.sort('hv2d_proxy', descending=True)['mean_phi_A'].to_numpy()
n_d      = dp_df.sort('hv2d_proxy', descending=True)['n_total'].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: hypervolume proxy per donor (bar chart)
ax = axes[0]
bar_colors = plt.cm.Blues(0.4 + 0.5 * hv_d / hv_d.max())
ax.bar(range(len(donors_sorted)), hv_d, color=bar_colors, edgecolor='white')
ax.set_xticks(range(len(donors_sorted)))
ax.set_xticklabels(donors_sorted, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('2D Hypervolume proxy (Φ_S–Φ_A)')
ax.set_title('Per-donor Pareto front quality\n(inter-individual variation)')

# Add n labels
for i, (hv, n_seq) in enumerate(zip(hv_d, n_d)):
    ax.text(i, hv + 0.001, f'n={n_seq//1000}K', ha='center', va='bottom', fontsize=6)

# Right: front centroids scatter
ax2 = axes[1]
for iso_color_map in donor_front_data.items():
    d_name, fc = iso_color_map
    ax2.scatter(fc[:, 0], fc[:, 1], s=1, alpha=0.2, rasterized=True)

# Overlay donor centroids
for row in dp_df.iter_rows(named=True):
    ax2.scatter(row['mean_phi_S'], row['mean_phi_A'], s=80,
                zorder=5, label=row['donor'], edgecolors='black', linewidth=0.5)
ax2.set_xlabel('Mean Φ_S on front')
ax2.set_ylabel('Mean Φ_A on front')
ax2.set_title('Per-donor front centroids\n(Φ_S vs Φ_A projection)')
ax2.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig(FIGURES / "fig_p4_donor_fronts.png", dpi=150, bbox_inches='tight')

dp_df.select(['donor', 'n_total', 'n_front', 'hv2d_proxy',
              'mean_phi_A', 'mean_phi_S', 'mean_phi_R']).write_csv(
    FIGURES / "fig_p4_donor_fronts.csv"
)

plt.show()
print("Saved.")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("PARETO FRONT SUMMARY")
print("=" * 60)

print(f"\nP1 Global (stratified sample n={sample_df.height:,}):")
print(f"   Pareto rank-1: {n_front:,} sequences ({100*n_front/sample_df.height:.1f}%)")

print(f"\nP2 Per-germline ({len(germline_pareto)} germlines, ≥{MIN_N_GERM} sequences):")
print(f"   HV proxy: mean={gp_df['hv2d_proxy'].mean():.4f}  "
      f"std={gp_df['hv2d_proxy'].std():.4f}")
print(f"   Best germline (highest HV): {gp_df[0]['v_gene'][0]}")

print(f"\nP3 Per-isotype:")
for row in ip_df.sort('hv2d_proxy', descending=True).iter_rows(named=True):
    print(f"   {row['isotype']:4s}: HV={row['hv2d_proxy']:.4f}  "
          f"front mean Φ_A={row['mean_phi_A']:.3f}  "
          f"front mean Φ_S={row['mean_phi_S']:.3f}")

print(f"\nP4 Per-donor ({len(donor_pareto)} donors):")
print(f"   HV proxy: mean={dp_df['hv2d_proxy'].mean():.4f}  "
      f"std={dp_df['hv2d_proxy'].std():.4f}")

# Biological flag
iso_hv = {r['isotype']: r['hv2d_proxy'] for r in isotype_pareto}
if iso_hv.get('IgG', 0) > iso_hv.get('IgM', 0):
    print("\n✓ IgG front HV > IgM front HV: maturation expands the Pareto front (expected).")
if iso_hv.get('IgE', 0) > iso_hv.get('IgG', 0):
    print("✓ IgE front HV > IgG front HV: IgE represents deepest maturation.")
if iso_hv.get('IgA', 0) < iso_hv.get('IgG', 0):
    print("⚠ IgA front HV < IgG: consistent with Step 1 V4 finding (IgA mixes\n"
          "  T-independent lower-SHM sequences → compressed front).")

## Step 6 Summary

| Calculation | Output table | Output figure(s) |
|-------------|-------------|------------------|
| P0: objective triplets | `objective_triplets.parquet` | — |
| P1: global Pareto front (sample) | `pareto_sample.csv` | `fig_p1_pareto_projections.png` |
| P2: per-germline fronts | `pareto_by_germline.csv` | `fig_p2_germline_hypervolume.png`, `fig_p2_germline_fronts.png` |
| P3: per-isotype fronts | `pareto_by_isotype.csv` | `fig_p3_isotype_fronts.png` |
| P4: per-donor fronts | `pareto_by_donor.csv` | `fig_p4_donor_fronts.png` |

**Key interpretations:**

1. **Global Pareto front**: The non-dominated sequences represent the efficient frontier — they achieve the best affinity improvement without incurring disproportionate structural or reactivity penalties. The front fraction (~X%) reflects the diversity of maturation outcomes in the population.

2. **Per-germline fronts (P2)**: Different IGHV genes explore different regions of the Pareto front, reflecting their distinct structural and chemical properties. BnAb germlines (IGHV1-2, IGHV1-69) are expected to have fronts extending further into the low-Φ_R region (stricter reactivity tolerance).

3. **Isotype fronts (P3)**: The progressive front expansion IgM → IgG/IgE confirms the maturation hypothesis. IgA may show compression due to T-independent B cells (Step 1 V4 finding).

4. **Donor fronts (P4)**: Inter-individual variation in front shape reflects genetic and environmental differences in the B cell tolerance landscape.

**Approximations and limitations:**
- P1 uses a stratified random sample (200K) for the global front — not all 1.46M sequences.
- Φ_S uses region-mean omega (not position-specific mutation calls): slightly coarse.
- The 2D hypervolume proxy (Φ_S–Φ_A projection) ignores the Φ_R dimension; a full 3D HV requires `pygmo`.
- Ties in objectives are treated as dominated (conservative).

**Next step:** `07_dynamics.ipynb` — Hamiltonian dynamics: trace lineage trajectories through (Φ_S, Φ_A, Φ_R) space and test whether they approach the Pareto front predictably.